# S3 / MinIO / Yandex Object Storage: boto3, AWS CLI и Spark

Ноутбук для лекционного демо. Показываем S3-compatible storage на примере MinIO, затем объясняем, как тот же подход переносится на Yandex Object Storage.

Что будет в ноутбуке:

- базовая терминология: bucket, object, key, prefix, endpoint;
- основные операции через `boto3`;
- те же операции через AWS CLI в стиле `!aws ...`;
- чтение CSV и запись Parquet через Spark `s3a://`;
- cleanup тестовых объектов.

Важно: реальные секреты не храним в notebook. Все credentials берём из переменных окружения.

## 1. Конфигурация

Для локального MinIO внутри Docker-сети используем endpoint `http://minio:9000`.

В текущем проекте MinIO настроен так:

- Console UI: `http://localhost:9001`;
- API внутри Docker: `http://minio:9000`;
- API с хоста: `http://localhost:9000`;
- login/password: `admin` / `password123`;
- bucket для демо: `datalake`.

Для Yandex Object Storage достаточно поменять env-переменные:

```bash
S3_ENDPOINT=https://storage.yandexcloud.net
S3_ACCESS_KEY=<YANDEX_STATIC_KEY_ID>
S3_SECRET_KEY=<YANDEX_STATIC_SECRET>
S3_BUCKET=<YANDEX_BUCKET_NAME>
```

In [1]:
import os
from pathlib import Path
from textwrap import dedent

# Defaults подходят для MinIO в docker-compose этого проекта.
S3_ENDPOINT = os.getenv('S3_ENDPOINT') or os.getenv('S3_ENDPOINT_URL') or 'http://minio:9000'
S3_ACCESS_KEY = os.getenv('S3_ACCESS_KEY', 'admin')
S3_SECRET_KEY = os.getenv('S3_SECRET_KEY', 'password123')
S3_BUCKET = os.getenv('S3_BUCKET', 'datalake')
S3_REGION = os.getenv('S3_REGION', 'us-east-1')
S3_PATH_STYLE_ACCESS = os.getenv('S3_PATH_STYLE_ACCESS', 'true')

RAW_KEY = 'raw/orders/dt=2026-06-23/orders.csv'
ODS_PREFIX = 'ods/orders/dt=2026-06-23/'
STAGING_PREFIX = 'staging/s3_notebook_demo/'
LOCAL_ORDERS = Path('/tmp/orders.csv')
LOCAL_DOWNLOAD = Path('/tmp/orders_downloaded.csv')

# Shell-команды !aws наследуют env текущего kernel process.
os.environ['AWS_ACCESS_KEY_ID'] = S3_ACCESS_KEY
os.environ['AWS_SECRET_ACCESS_KEY'] = S3_SECRET_KEY
os.environ['AWS_DEFAULT_REGION'] = S3_REGION
os.environ['S3_ENDPOINT'] = S3_ENDPOINT
os.environ['S3_BUCKET'] = S3_BUCKET
os.environ['RAW_KEY'] = RAW_KEY
os.environ['ODS_PREFIX'] = ODS_PREFIX
os.environ['LOCAL_ORDERS'] = str(LOCAL_ORDERS)

print('S3_ENDPOINT =', S3_ENDPOINT)
print('S3_BUCKET   =', S3_BUCKET)
print('RAW_KEY     =', RAW_KEY)

S3_ENDPOINT = http://minio:9000
S3_BUCKET   = datalake
RAW_KEY     = raw/orders/dt=2026-06-23/orders.csv


## 2. Готовим маленький CSV

Для лекции удобнее создать файл прямо из notebook. Так демо не зависит от того, какая директория смонтирована в Jupyter-контейнер.

In [2]:
orders_csv = dedent('''\
order_id,user_id,status,amount,created_at,dt
1001,501,created,1250.50,2026-06-23 09:15:00,2026-06-23
1002,502,paid,3200.00,2026-06-23 09:20:00,2026-06-23
1003,503,paid,790.99,2026-06-23 09:45:00,2026-06-23
1004,501,shipped,1250.50,2026-06-23 10:05:00,2026-06-23
1005,504,cancelled,450.00,2026-06-23 10:20:00,2026-06-23
1006,505,created,1999.90,2026-06-23 10:35:00,2026-06-23
1007,506,paid,8750.00,2026-06-23 11:00:00,2026-06-23
1008,507,paid,150.75,2026-06-23 11:10:00,2026-06-23
1009,508,shipped,4320.40,2026-06-23 11:25:00,2026-06-23
1010,509,created,999.00,2026-06-23 11:50:00,2026-06-23
''')

LOCAL_ORDERS.write_text(orders_csv, encoding='utf-8')
print(LOCAL_ORDERS)
print(LOCAL_ORDERS.read_text(encoding='utf-8'))

/tmp/orders.csv
order_id,user_id,status,amount,created_at,dt
1001,501,created,1250.50,2026-06-23 09:15:00,2026-06-23
1002,502,paid,3200.00,2026-06-23 09:20:00,2026-06-23
1003,503,paid,790.99,2026-06-23 09:45:00,2026-06-23
1004,501,shipped,1250.50,2026-06-23 10:05:00,2026-06-23
1005,504,cancelled,450.00,2026-06-23 10:20:00,2026-06-23
1006,505,created,1999.90,2026-06-23 10:35:00,2026-06-23
1007,506,paid,8750.00,2026-06-23 11:00:00,2026-06-23
1008,507,paid,150.75,2026-06-23 11:10:00,2026-06-23
1009,508,shipped,4320.40,2026-06-23 11:25:00,2026-06-23
1010,509,created,999.00,2026-06-23 11:50:00,2026-06-23



## 3. Подключение через boto3

`boto3` - основной Python SDK для S3 API. Для S3-compatible storage важно явно указать `endpoint_url`. Без endpoint клиент будет пытаться ходить в AWS S3.

In [3]:
import boto3
from botocore.client import Config
from botocore.exceptions import ClientError

s3 = boto3.client(
    's3',
    endpoint_url=S3_ENDPOINT,
    aws_access_key_id=S3_ACCESS_KEY,
    aws_secret_access_key=S3_SECRET_KEY,
    region_name=S3_REGION,
    config=Config(signature_version='s3v4', s3={'addressing_style': 'path'}),
)

s3

## 4. Bucket operations через boto3

Основные методы:

- `list_buckets()` - показать buckets, доступные credentials;
- `create_bucket()` - создать bucket;
- `head_bucket()` - проверить, что bucket существует и доступен.

In [4]:
response = s3.list_buckets()
[bucket['Name'] for bucket in response.get('Buckets', [])]

['datalake']

In [ ]:
try:
    s3.create_bucket(Bucket=S3_BUCKET)
    print(f'Bucket {S3_BUCKET!r} created')
except ClientError as e:
    code = e.response.get('Error', {}).get('Code')
    if code in {'BucketAlreadyOwnedByYou', 'BucketAlreadyExists'}:
        print(f'Bucket {S3_BUCKET!r} already exists')
    else:
        raise

s3.head_bucket(Bucket=S3_BUCKET)
print(f'Bucket {S3_BUCKET!r} is available')

## 5. Object write: `put_object()` и `upload_file()`

Есть два частых способа загрузки:

- `put_object()` - передать bytes/string прямо из памяти;
- `upload_file()` - загрузить локальный файл с диска.

In [ ]:
readme_key = STAGING_PREFIX + 'hello.txt'

s3.put_object(
    Bucket=S3_BUCKET,
    Key=readme_key,
    Body='Hello from boto3 and MinIO!\n'.encode('utf-8'),
    ContentType='text/plain',
)

print('uploaded:', f's3://{S3_BUCKET}/{readme_key}')

In [ ]:
s3.upload_file(str(LOCAL_ORDERS), S3_BUCKET, RAW_KEY)
print('uploaded:', f's3://{S3_BUCKET}/{RAW_KEY}')

## 6. Object listing: `list_objects_v2()` и paginator

В S3 нет настоящих директорий. `raw/orders/` - это prefix ключей. Listing по prefix показывает объекты, ключи которых начинаются с этой строки.

In [ ]:
response = s3.list_objects_v2(Bucket=S3_BUCKET, Prefix='raw/orders/')

for obj in response.get('Contents', []):
    print(obj['Key'], obj['Size'], obj['LastModified'])

In [ ]:
paginator = s3.get_paginator('list_objects_v2')

for page in paginator.paginate(Bucket=S3_BUCKET, Prefix='raw/'):
    for obj in page.get('Contents', []):
        print(obj['Key'])

## 7. Object metadata: `head_object()`

`head_object()` возвращает метаданные без скачивания тела объекта: размер, content type, etag, last modified.

In [ ]:
metadata = s3.head_object(Bucket=S3_BUCKET, Key=RAW_KEY)

{
    'ContentLength': metadata['ContentLength'],
    'ContentType': metadata.get('ContentType'),
    'ETag': metadata.get('ETag'),
    'LastModified': metadata.get('LastModified'),
}

## 8. Object read: `get_object()` и `download_file()`

`get_object()` удобен для чтения небольшого объекта в память. `download_file()` скачивает объект в локальный файл.

In [ ]:
obj = s3.get_object(Bucket=S3_BUCKET, Key=RAW_KEY)
text = obj['Body'].read().decode('utf-8')
print(text[:300])

In [ ]:
s3.download_file(S3_BUCKET, RAW_KEY, str(LOCAL_DOWNLOAD))
print(LOCAL_DOWNLOAD)
print(LOCAL_DOWNLOAD.read_text(encoding='utf-8').splitlines()[:3])

## 9. Copy, tags и presigned URL

Полезные методы:

- `copy_object()` - скопировать объект внутри bucket или между buckets;
- `put_object_tagging()` / `get_object_tagging()` - теги объекта;
- `generate_presigned_url()` - временная ссылка на объект.

In [ ]:
copy_key = STAGING_PREFIX + 'orders_copy.csv'

s3.copy_object(
    Bucket=S3_BUCKET,
    Key=copy_key,
    CopySource={'Bucket': S3_BUCKET, 'Key': RAW_KEY},
)

print('copied to:', f's3://{S3_BUCKET}/{copy_key}')

In [ ]:
s3.put_object_tagging(
    Bucket=S3_BUCKET,
    Key=copy_key,
    Tagging={
        'TagSet': [
            {'Key': 'course', 'Value': 'data-engineering'},
            {'Key': 'purpose', 'Value': 'lecture-demo'},
        ]
    },
)

s3.get_object_tagging(Bucket=S3_BUCKET, Key=copy_key)['TagSet']

In [ ]:
url = s3.generate_presigned_url(
    ClientMethod='get_object',
    Params={'Bucket': S3_BUCKET, 'Key': RAW_KEY},
    ExpiresIn=300,
)

print(url)

## 10. Обработка ошибок boto3

S3 API часто возвращает ошибки вроде `NoSuchKey`, `NoSuchBucket`, `AccessDenied`, `SignatureDoesNotMatch`. В `boto3` они приходят как `ClientError`.

In [ ]:
try:
    s3.head_object(Bucket=S3_BUCKET, Key='raw/orders/missing.csv')
except ClientError as e:
    error = e.response.get('Error', {})
    print('Code:', error.get('Code'))
    print('Message:', error.get('Message'))

## 11. AWS CLI из notebook: `!aws ...`

В HDFS-лекции мы показывали команды через `!hdfs dfs ...`. Для S3 можно сделать так же через `!aws ...`.

Это не Python API, а shell-команды Jupyter. Они удобны для демонстрации того же интерфейса, которым часто пользуются инженеры в терминале.

В Docker-образ Jupyter/Spark для этого проекта добавлен `awscli`, поэтому после пересборки стенда команда `aws` доступна прямо из notebook.

In [1]:
!aws --version

/usr/bin/sh: 1: aws: not found


In [ ]:
!aws --endpoint-url $S3_ENDPOINT s3 ls

In [ ]:
# Создать bucket. Если bucket уже есть, команда может вернуть ошибку, это нормально для демо.
!aws --endpoint-url $S3_ENDPOINT s3 mb s3://$S3_BUCKET || true

In [ ]:
!aws --endpoint-url $S3_ENDPOINT s3 cp $LOCAL_ORDERS s3://$S3_BUCKET/$RAW_KEY

In [ ]:
!aws --endpoint-url $S3_ENDPOINT s3 ls s3://$S3_BUCKET/raw/orders/dt=2026-06-23/

In [ ]:
!aws --endpoint-url $S3_ENDPOINT s3api head-object --bucket datalake --key /raw/orders/dt=2026-06-23/orders.csv

In [ ]:
!aws --endpoint-url $S3_ENDPOINT s3api get-object --bucket datalake --key /raw/orders/dt=2026-06-23/orders.csv /dev/stdout

## 12. Spark читает CSV из S3 и пишет Parquet обратно

Spark работает с S3-compatible storage через Hadoop connector `s3a://`.

Важные настройки:

- `fs.s3a.endpoint` - endpoint MinIO/Yandex Object Storage;
- `fs.s3a.access.key` и `fs.s3a.secret.key`;
- `fs.s3a.path.style.access=true` для MinIO;
- `fs.s3a.impl=org.apache.hadoop.fs.s3a.S3AFileSystem`;
- jar dependencies `hadoop-aws` и AWS SDK bundle должны быть доступны Spark.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName('lecture-05-s3-notebook-demo')
    .config('spark.sql.shuffle.partitions', '2')
    .config('spark.jars.packages', 'org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262')
    .config('spark.hadoop.fs.s3a.endpoint', S3_ENDPOINT)
    .config('spark.hadoop.fs.s3a.access.key', S3_ACCESS_KEY)
    .config('spark.hadoop.fs.s3a.secret.key', S3_SECRET_KEY)
    .config('spark.hadoop.fs.s3a.path.style.access', S3_PATH_STYLE_ACCESS)
    .config('spark.hadoop.fs.s3a.impl', 'org.apache.hadoop.fs.s3a.S3AFileSystem')
    .config('spark.hadoop.fs.s3a.connection.ssl.enabled', str(S3_ENDPOINT.startswith('https://')).lower())
    .config('spark.hadoop.fs.s3a.aws.credentials.provider', 'org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider')
    .getOrCreate()
)

spark.sparkContext.setLogLevel('WARN')

Если SparkSession уже был создан раньше без `hadoop-aws`, остановите kernel и запустите notebook сначала. `spark.jars.packages` должен примениться до создания SparkContext.

In [ ]:
raw_path = f's3a://{S3_BUCKET}/{RAW_KEY}'
ods_path = f's3a://{S3_BUCKET}/{ODS_PREFIX}'

print(raw_path)
print(ods_path)

In [ ]:
raw_orders = spark.read.option('header', 'true').csv(raw_path)
raw_orders.show(truncate=False)
raw_orders.printSchema()

In [ ]:
orders = (
    raw_orders.select(
        F.col('order_id').cast('long').alias('order_id'),
        F.col('user_id').cast('long').alias('user_id'),
        F.col('status'),
        F.col('amount').cast('double').alias('amount'),
        F.to_timestamp('created_at', 'yyyy-MM-dd HH:mm:ss').alias('created_at'),
        F.to_date('dt', 'yyyy-MM-dd').alias('dt'),
    )
    .withColumn('loaded_at', F.current_timestamp())
)

orders.show(truncate=False)
orders.printSchema()

In [ ]:
# Для лекции coalesce(1) делает результат проще для просмотра в MinIO UI.
# В production это не универсальная рекомендация: можно создать bottleneck на одном executor.
orders.coalesce(1).write.mode('overwrite').parquet(ods_path)
print('written:', ods_path)

In [ ]:
written_orders = spark.read.parquet(ods_path)
written_orders.orderBy('order_id').show(truncate=False)
written_orders.printSchema()

## 13. Что важно про Spark и S3 committers

Spark пишет результат не одним магическим файлом. Обычно каждая task пишет свои part-файлы. Для файловых систем типа HDFS классический commit часто использует rename временных файлов в финальный каталог.

В S3/Object Storage rename не является дешевой атомарной операцией как в HDFS. Обычно это copy + delete. Поэтому для больших jobs стандартные file commit protocols могут быть медленными или давать неожиданные эффекты при сбоях.

Что нужно знать на уровне лекции:

- на S3 нужно аккуратно выбирать output path;
- `mode('overwrite')` может удалить весь указанный prefix;
- много маленьких файлов ухудшает запись и чтение;
- для production часто настраивают S3A committers или используют table formats вроде Iceberg/Delta/Hudi;
- конкретные настройки зависят от версии Spark/Hadoop и облака.

В учебном демо мы оставляем стандартный write path, потому что цель - показать базовую механику `s3a://`, а не production tuning.

## 14. Cleanup

Удалим тестовые объекты, которые создали в notebook. Bucket оставляем, потому что его использует учебный стенд.

In [ ]:
# delete_object удаляет один объект.
s3.delete_object(Bucket=S3_BUCKET, Key=readme_key)
print('deleted:', readme_key)

In [ ]:
# delete_objects удаляет несколько объектов одним запросом.
keys_to_delete = [
    {'Key': copy_key},
]

s3.delete_objects(Bucket=S3_BUCKET, Delete={'Objects': keys_to_delete, 'Quiet': False})

In [ ]:
# Удалить raw/ods демо-данные через AWS CLI. Раскомментируйте, если нужно полностью очистить demo prefix.
# !aws --endpoint-url $S3_ENDPOINT s3 rm s3://$S3_BUCKET/raw/orders/dt=2026-06-23/ --recursive
# !aws --endpoint-url $S3_ENDPOINT s3 rm s3://$S3_BUCKET/ods/orders/dt=2026-06-23/ --recursive

## 15. Переключение на Yandex Object Storage

Перед запуском notebook задайте переменные окружения для Jupyter-контейнера или kernel environment:

```bash
S3_ENDPOINT=https://storage.yandexcloud.net
S3_ACCESS_KEY=<YANDEX_STATIC_KEY_ID>
S3_SECRET_KEY=<YANDEX_STATIC_SECRET>
S3_BUCKET=<YANDEX_BUCKET_NAME>
S3_REGION=ru-central1
S3_PATH_STYLE_ACCESS=true
```

Не вставляйте реальные секреты в notebook и не показывайте secret key на записи. После демо удалите тестовые объекты, bucket и static key, если они создавались только для занятия.